# Notebook D — Statistics, Figures, Tables, Repo Bundle

Consumes `runs.csv` + `preds/` and emits every artefact the paper and the GitHub repo need,
each tagged with the section it belongs to. Nothing here needs a GPU.

In [ ]:
import json, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from scipy.stats import chi2
from sklearn.metrics import f1_score, confusion_matrix, classification_report

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='paper', font_scale=1.05)
plt.rcParams['savefig.dpi'] = 300

RES = Path('/kaggle/input/nbc-runs')
AUD = Path('/kaggle/input/nba-audit-splits')
OUT = Path('/kaggle/working'); OUT.mkdir(exist_ok=True, parents=True)

runs = pd.read_csv(RES / 'runs.csv')
CFG  = json.loads((RES / 'experiment_config.json').read_text())
SEEDS = CFG['seeds']
print(f'{len(runs)} runs | tasks {sorted(runs.task.unique())}')

class Artifacts:
    def __init__(self, root):
        self.root = Path(root)
        (self.root/'figures').mkdir(parents=True, exist_ok=True)
        (self.root/'tables').mkdir(parents=True, exist_ok=True)
        self.p = self.root/'index.json'
        self.items = json.loads(self.p.read_text()) if self.p.exists() else []
    def _put(self, r):
        self.items = [i for i in self.items if i['name'] != r['name']] + [r]
        self.p.write_text(json.dumps(self.items, indent=2))
    def figure(self, fig, name, caption, section):
        for e in ('png','pdf'): fig.savefig(self.root/'figures'/f'{name}.{e}', bbox_inches='tight')
        self._put({'kind':'figure','name':name,'caption':caption,'section':section})
        print(f'  [artifact] figure {name} -> {section}')
    def table(self, df, name, caption, section):
        df.to_csv(self.root/'tables'/f'{name}.csv', index=False)
        (self.root/'tables'/f'{name}.tex').write_text(
            df.to_latex(index=False, escape=False, float_format='%.4f',
                        caption=caption, label=f'tab:{name}'))
        self._put({'kind':'table','name':name,'caption':caption,'section':section})
        print(f'  [artifact] table  {name} -> {section}')
    def readme(self):
        L = ['# Paper assets','','| Section | Kind | Name | Caption |','|---|---|---|---|']
        for i in sorted(self.items, key=lambda r:(str(r['section']), r['name'])):
            L.append(f"| {i['section']} | {i['kind']} | `{i['name']}` | {i['caption'][:110]} |")
        (self.root/'README.md').write_text('\n'.join(L))
        print(f'\n{len(self.items)} artifacts indexed')

ART = Artifacts(OUT/'paper_assets')

def preds(run_id):
    z = np.load(RES/'preds'/f"{run_id.replace('|','_')}.npz")
    return z['pred'], z['label']

In [ ]:
# ── Aggregate over seeds ────────────────────────────────────────────────
def summarise(r):
    out = []
    for (t, c, lab, fam, pr), g in r.groupby(['task','condition','label','family','protocol']):
        d = {'task': t, 'condition': c, 'label': lab, 'family': fam, 'protocol': pr,
             'meta_dim': int(g.meta_dim.iloc[0]), 'n_seeds': len(g)}
        for m in ['test_acc','test_macro_f1','test_bal_acc']:
            v = g[m].values; n = len(v); sd = v.std(ddof=1) if n > 1 else 0.
            h = stats.t.ppf(.975, n-1)*sd/np.sqrt(n) if n > 1 else 0.
            d |= {f'{m}_mean': v.mean(), f'{m}_sd': sd, f'{m}_lo': v.mean()-h, f'{m}_hi': v.mean()+h}
        out.append(d)
    return pd.DataFrame(out)

summ = summarise(runs)
mainB = summ.query('task=="B" and protocol=="spatial" and family!="quality"').sort_values('condition')
mainA = summ.query('task=="A" and protocol=="spatial" and family!="quality"').sort_values('condition')
for nm, t in [('B', mainB), ('A', mainA)]:
    print(f'\n--- Task {nm} ---')
    display(t[['condition','label','family','meta_dim','test_acc_mean','test_acc_sd',
               'test_macro_f1_mean','test_macro_f1_sd']].round(4))
ART.table(mainB.round(4), 'tab3_main_taskB',
          'Task B (scene): test performance under the spatially-blocked split, mean +/- s.d. over 5 seeds.', '4.2')
ART.table(mainA.round(4), 'tab4_main_taskA',
          'Task A (continent): the degenerate formulation, shown for contrast.', '4.1')

In [ ]:
# ════════════════════════════════════════════════════════════════════════
#  SYNERGY & LEAKAGE DECOMPOSITION — the paper's core table
# ════════════════════════════════════════════════════════════════════════
PAIR = {'C4':'M0', 'C5':'M1', 'C1':'M2', 'O1':'M3'}
rows = []
for t in ['A','B']:
    for pr in CFG['protocols']:
        base = runs[(runs.task == t) & (runs.protocol == pr)]
        def get(c, _b=base):
            v = _b[_b.condition == c].test_acc
            return v.mean() if len(v) else np.nan
        ai = get('C0')
        for f, m in PAIR.items():
            af, am = get(f), get(m)
            if np.isnan(af) or np.isnan(am): continue
            rows.append({'task': t, 'protocol': pr, 'fusion': f, 'ctrl': m,
                         'Acc_image': ai, 'Acc_meta': am, 'Acc_fusion': af,
                         'naive_delta': af-ai, 'synergy_S': af-max(ai, am),
                         'leakage_L': am/af if af else np.nan})
syn = pd.DataFrame(rows)
syn['verdict'] = np.where(syn.leakage_L > .95, 'LEAKAGE',
                  np.where(syn.synergy_S > .01, 'genuine synergy',
                  np.where(syn.synergy_S > -.01, 'no gain', 'metadata harms')))
display(syn.round(4))
ART.table(syn.round(4), 'tab5_synergy_leakage',
          'Decomposition of apparent metadata gains into leakage, capacity and synergy. '
          'Only S>0 supports a complementarity claim.', '4.3')

In [ ]:
# ── Paired significance vs image-only ───────────────────────────────────
def paired_boot(t, a, b, pr, nb=5000, seed=0):
    rng = np.random.RandomState(seed); A, B = [], []
    for s in SEEDS:
        try:
            pa, l = preds(f'{t}|{a}|{pr}|s{s}'); pb, _ = preds(f'{t}|{b}|{pr}|s{s}')
        except FileNotFoundError: continue
        A.append(pa == l); B.append(pb == l)
    if not A: return np.nan, (np.nan, np.nan), np.nan
    d = np.vstack(A).mean(0) - np.vstack(B).mean(0); n = len(d)
    bt = np.array([d[rng.randint(0, n, n)].mean() for _ in range(nb)])
    return d.mean(), (np.percentile(bt, 2.5), np.percentile(bt, 97.5)), \
           min(2*min((bt <= 0).mean(), (bt >= 0).mean()), 1.0)

def modal(t, c, pr):
    st, l = [], None
    for s in SEEDS:
        try: p, l = preds(f'{t}|{c}|{pr}|s{s}'); st.append(p)
        except FileNotFoundError: pass
    if not st: return None, None
    return np.asarray(stats.mode(np.vstack(st), axis=0, keepdims=False).mode).ravel(), l

def mcnemar(a, b, y):
    ca, cb = a == y, b == y
    B, C = int((ca & ~cb).sum()), int((~ca & cb).sum())
    if B+C == 0: return 1.0
    return float(1 - chi2.cdf((abs(B-C)-1)**2/(B+C), 1))

def holm(p):
    p = np.asarray(p, float); adj = np.full_like(p, np.nan); ok = ~np.isnan(p)
    idx = np.where(ok)[0][np.argsort(p[ok])]; m, run = len(idx), 0.
    for r, i in enumerate(idx):
        run = max(run, min((m-r)*p[i], 1.)); adj[i] = run
    return adj

def d_cohen(t, a, b, pr):
    sel = (runs.task == t) & (runs.protocol == pr)
    x = runs[sel & (runs.condition == a)].test_macro_f1.values
    y = runs[sel & (runs.condition == b)].test_macro_f1.values
    if len(x) < 2 or len(y) < 2: return np.nan
    s = np.sqrt(((len(x)-1)*x.var(ddof=1)+(len(y)-1)*y.var(ddof=1))/(len(x)+len(y)-2))
    return (x.mean()-y.mean())/s if s > 0 else np.nan

sig_all = []
for t in ['B','A']:
    _s = summ[(summ.task == t) & (summ.protocol == 'spatial') & (summ.family != 'quality')]
    tg = [c for c in _s.condition if c != 'C0']
    rr = []
    for c in tg:
        dl, ci, p = paired_boot(t, c, 'C0', 'spatial')
        pa, y = modal(t, c, 'spatial'); pb, _ = modal(t, 'C0', 'spatial')
        rr.append({'task': t, 'condition': c, 'delta_acc': dl, 'ci_lo': ci[0], 'ci_hi': ci[1],
                   'p_boot': p, 'p_mcnemar': mcnemar(pa, pb, y) if pa is not None else np.nan,
                   'cohens_d': d_cohen(t, c, 'C0', 'spatial')})
    s = pd.DataFrame(rr)
    s['p_boot_holm'] = holm(s.p_boot.values); s['p_mcnemar_holm'] = holm(s.p_mcnemar.values)
    s['significant'] = (s.p_boot_holm < .05) & (np.sign(s.ci_lo) == np.sign(s.ci_hi))
    sig_all.append(s)
sig = pd.concat(sig_all, ignore_index=True)
display(sig.round(4))
ART.table(sig.round(4), 'tab6_significance',
          'Paired comparison against the image-only baseline; 95% paired-bootstrap CI and '
          'Holm-corrected p-values across all comparisons within each task.', '4.4')

In [ ]:
# ════════ FIGURE: ablation with CIs, both tasks ═════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(13, 5.4))
pal = {'ablation':'#2980b9','oracle':'#c0392b','control':'#7f8c8d'}
for ax, (t, d) in zip(axes, [('A', mainA), ('B', mainB)]):
    d = d.sort_values('test_macro_f1_mean')
    err = [d.test_macro_f1_mean-d.test_macro_f1_lo, d.test_macro_f1_hi-d.test_macro_f1_mean]
    ax.barh(np.arange(len(d)), d.test_macro_f1_mean, xerr=err,
            color=[pal[f] for f in d.family], error_kw=dict(ecolor='k', capsize=3, lw=1))
    ax.set_yticks(np.arange(len(d))); ax.set_yticklabels(d.condition + '  ' + d.label, fontsize=8)
    b = d.query('condition=="C0"').test_macro_f1_mean
    if len(b): ax.axvline(b.iloc[0], ls='--', c='k', lw=1.2)
    ax.set_xlabel('test macro-F1'); ax.set_title(f'Task {t}', fontweight='bold')
fig.suptitle('Ablation, oracles and controls (spatially-blocked split, 5 seeds, 95% CI)',
             fontweight='bold')
plt.tight_layout()
ART.figure(fig, 'fig3_ablation_both_tasks',
           'Ablation with oracle upper bounds and permutation controls for both tasks.', '4.2')
plt.show()

In [ ]:
# ════════ FIGURE: leakage diagnostic scatter ════════════════════════════
fig, ax = plt.subplots(figsize=(7, 6.2))
mk = {'A':'s','B':'o'}
for _, r in syn.query('protocol=="spatial"').iterrows():
    ax.scatter(r.Acc_meta, r.Acc_fusion, s=150, marker=mk[r.task],
               color='#c0392b' if r.leakage_L > .95 else '#2980b9', zorder=3,
               edgecolor='k', lw=.6)
    ax.annotate(f'{r.task}:{r.fusion}', (r.Acc_meta, r.Acc_fusion),
                textcoords='offset points', xytext=(9, -3), fontsize=9)
ax.plot([0, 1.02], [0, 1.02], ls=':', c='k', lw=1, label='fusion = metadata alone (leakage)')
ax.fill_between([0, 1.02], [0, 1.02], 1.02, color='#2980b9', alpha=.07)
ax.text(.30, .93, 'genuine synergy region', color='#2980b9', fontsize=9)
ax.set_xlim(0, 1.02); ax.set_ylim(0, 1.02)
ax.set_xlabel('accuracy from metadata alone'); ax.set_ylabel('accuracy from image + metadata')
ax.set_title('Points on the diagonal are not fusion results', fontweight='bold')
ax.legend(loc='lower right', fontsize=8.5)
plt.tight_layout()
ART.figure(fig, 'fig4_leakage_diagnostic',
           'Fusion accuracy against metadata-only accuracy. Task A sits on the diagonal '
           '(leakage); Task B sits in the synergy region.', '4.3')
plt.show()

In [ ]:
# ════════ FIGURE: dose-response, four axes ══════════════════════════════
q = runs.query('family=="quality"')
AX = [('missing','fraction of metadata cells missing',0),
      ('noise','additive noise sigma (train-std units)',0),
      ('misalign','fraction of records misaligned',0),
      ('precision','GPS cell size (degrees)',1)]
imgf1 = mainB.query('condition=="C0"').test_macro_f1_mean.iloc[0]
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
tolr = []
for ax, (a, xl, lg) in zip(axes.ravel(), AX):
    s = q[q.axis == a]
    if s.empty: ax.axis('off'); continue
    g = s.groupby('level').test_macro_f1.agg(['mean','std','count']).reset_index()
    h = stats.t.ppf(.975, g['count']-1)*g['std']/np.sqrt(g['count'])
    ax.errorbar(g.level, g['mean'], yerr=h, marker='o', capsize=3, lw=2, color='#2980b9')
    ax.axhline(imgf1, ls='--', c='k', lw=1.2, label='image only')
    if lg: ax.set_xscale('log')
    ax.set_xlabel(xl); ax.set_ylabel('test macro-F1')
    ax.set_title(f'axis: {a}', fontweight='bold', fontsize=10); ax.legend(fontsize=8)
    x = np.log10(g.level.replace(0, np.nan)) if lg else g.level
    m = ~pd.Series(x).isna()
    if m.sum() > 2:
        sl, ic, r, p, se = stats.linregress(np.asarray(x)[m], g['mean'][m])
        ax.text(.03, .06, f'slope={sl:+.4f}\nr={r:+.3f}, p={p:.3g}', transform=ax.transAxes,
                fontsize=8, bbox=dict(fc='white', alpha=.85, ec='.7'))
    best = g['mean'].max(); ov = g[(best-g['mean']) > .01]
    tolr.append({'axis': a, 'best_f1': best, 'worst_f1': g['mean'].iloc[-1],
                 'total_drop': best-g['mean'].iloc[-1],
                 'first_level_losing_1pt': ov.level.iloc[0] if len(ov) else np.nan,
                 'still_beats_image_only': bool(g['mean'].iloc[-1] > imgf1)})
    ART.table(g.round(4), f'tabQ_{a}', f'Dose-response along the {a} axis (Task B, spatial).', '4.5')
fig.suptitle('Metadata quality dose-response (Task B, spatially-blocked, 5 seeds)',
             fontweight='bold', fontsize=12)
plt.tight_layout()
ART.figure(fig, 'fig5_dose_response',
           'Macro-F1 against four independent metadata-degradation axes.', '4.5')
plt.show()
tol = pd.DataFrame(tolr)
display(tol.round(4))
ART.table(tol.round(4), 'tab7_degradation_tolerance',
          'Sensitivity of Task B macro-F1 to each metadata degradation axis.', '4.5')

In [ ]:
# ════════ FIGURE: protocol inflation ════════════════════════════════════
pp = (summ.query('family in ["ablation","oracle"]')
        .pivot_table(index=['task','condition'], columns='protocol', values='test_macro_f1_mean'))
if {'random','spatial'} <= set(pp.columns):
    pp['inflation'] = pp['random'] - pp['spatial']
    fig, ax = plt.subplots(figsize=(9.5, 4.6))
    x = np.arange(len(pp))
    ax.bar(x-.2, pp['random'], .4, label='random', color='#c0392b')
    ax.bar(x+.2, pp['spatial'], .4, label='spatial', color='#2980b9')
    ax.set_xticks(x); ax.set_xticklabels([f'{a}:{b}' for a, b in pp.index], rotation=45, ha='right')
    ax.set_ylabel('test macro-F1'); ax.legend()
    ax.set_title('Random splitting inflates every condition', fontweight='bold')
    plt.tight_layout()
    ART.figure(fig, 'fig6_protocol_inflation',
               'Macro-F1 under random versus spatially-blocked splits.', '4.6')
    plt.show()
    display(pp.round(4).sort_values('inflation', ascending=False))
    ART.table(pp.reset_index().round(4), 'tab8_protocol_inflation',
              'Inflation attributable to random splitting.', '4.6')
    print(f"\nMean inflation: {pp['inflation'].mean():+.4f} macro-F1")

In [ ]:
# ════════ Per-class synergy + confusion (Task B) ════════════════════════
cls = CFG['classesB']
rowsc = []
for c in ['C0','C5']:
    p, y = modal('B', c, 'spatial')
    if p is None: continue
    for k, f in enumerate(f1_score(y, p, average=None, labels=range(len(cls)))):
        rowsc.append({'class': cls[k], 'condition': c, 'f1': f})
pc = pd.DataFrame(rowsc).pivot(index='class', columns='condition', values='f1')
if {'C0','C5'} <= set(pc.columns):
    pc['gain'] = pc['C5'] - pc['C0']
    pc = pc.sort_values('gain')
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].barh(pc.index, pc.gain, color=np.where(pc.gain > 0, '#2980b9', '#c0392b'))
    axes[0].axvline(0, c='k', lw=1)
    axes[0].set_xlabel('macro-F1 gain from metadata'); axes[0].set_title(
        'Per-class benefit: geographic classes gain most', fontweight='bold', fontsize=10)
    p, y = modal('B', 'C5', 'spatial')
    cm = confusion_matrix(y, p, labels=range(len(cls)))
    sns.heatmap(cm/np.maximum(cm.sum(1, keepdims=True), 1), annot=False, cmap='Blues',
                xticklabels=cls, yticklabels=cls, ax=axes[1], cbar=True)
    axes[1].set_xlabel('predicted'); axes[1].set_ylabel('true')
    axes[1].set_title('Confusion matrix (C5)', fontweight='bold', fontsize=10)
    plt.tight_layout()
    ART.figure(fig, 'fig7_perclass_confusion',
               'Per-class gain from metadata and confusion matrix for the best honest condition.', '4.7')
    plt.show()
    ART.table(pc.reset_index().round(4), 'tab9_perclass_gain',
              'Per-class macro-F1 gain from metadata (Task B).', '4.7')
    (OUT/'classification_report_C5.txt').write_text(
        classification_report(y, p, target_names=cls, digits=4))

In [ ]:
# ════════ Draft results paragraph + repo bundle ═════════════════════════
L = []
sB = syn.query('task=="B" and protocol=="spatial"').set_index('fusion')
sA = syn.query('task=="A" and protocol=="spatial"').set_index('fusion')
L.append(f"Task B image-only baseline: acc={mainB.query('condition==\"C0\"').test_acc_mean.iloc[0]:.4f}, "
         f"macro-F1={imgf1:.4f} (majority {CFG['majorityB']:.4f}).")
if 'C4' in sA.index:
    r = sA.loc['C4']
    L.append(f"Task A: GPS alone reaches acc={r.Acc_meta:.4f} (L={r.leakage_L:.3f}); the apparent "
             f"gain of {r.naive_delta:+.4f} collapses to synergy {r.synergy_S:+.4f}.")
if 'C5' in sB.index:
    r = sB.loc['C5']
    L.append(f"Task B: metadata-only acc={r.Acc_meta:.4f}, fusion acc={r.Acc_fusion:.4f}, "
             f"synergy S={r.synergy_S:+.4f} (L={r.leakage_L:.3f}).")
s5 = sig.query('task=="B" and condition=="C5"')
if len(s5):
    r = s5.iloc[0]
    L.append(f"Against image-only: delta={r.delta_acc:+.4f} [95% CI {r.ci_lo:+.4f},{r.ci_hi:+.4f}], "
             f"Holm p={r.p_boot_holm:.4g}, d={r.cohens_d:+.2f}.")
for _, r in tol.iterrows():
    L.append(f"{r.axis}: macro-F1 falls {r.total_drop:.4f} best-to-worst; first level costing "
             f">0.01 F1 is {r.first_level_losing_1pt}.")
txt = '\n\n'.join(L); print(txt)
(OUT/'results_paragraph_draft.txt').write_text(txt)

ART.readme()
runs.to_csv(OUT/'paper_assets'/'runs.csv', index=False)
json.dump(CFG, open(OUT/'paper_assets'/'experiment_config.json','w'), indent=2)
print('\npaper_assets/ contains every figure (png+pdf), table (csv+tex), '
      'the run log, the config, and an indexed README. Commit that folder to the repo.')
